In [1]:
import ipdb # <- трасировка и точки останова

from header import __root__
# Internal modules
from src import gs

🔑 Found password in password.txt (DEBUG MODE)
✅ Successfully opened KeePass database: C:\Users\user\Documents\repos\hypotez\secrets\credentials.kdbx
Failed to load GAPI credentials


In [2]:
import importlib
import os
import asyncio
import time
from pathlib import Path
from types import SimpleNamespace
from typing import Optional, List, Any
from dataclasses import dataclass, field


from src.suppliers.suppliers_list import *
from src.suppliers.get_graber_by_supplier  import get_graber_by_supplier_prefix, get_graber_by_supplier_url
from src.suppliers.graber import Graber
from src.webdriver.driverless import use_pydoll as pydoll_driver
from src.webdriver.firefox import Firefox
from src.webdriver.chrome import Chrome
from src.llm.gemini import GoogleGenerativeAi
from src.llm.openai.model import OpenAIModel
from src.endpoints.prestashop.product import PrestaProduct
from src.endpoints.prestashop.language import PrestaLanguage
from src.endpoints.prestashop.product_fields import ProductFields
from src.endpoints.advertisement.facebook.scenarios.post_message import (
    post_message,
)
from src.utils.file import read_text_file, save_text_file, get_filenames_from_directory

from src.utils.jjson import j_loads, j_loads_ns, j_dumps
from src.utils.image import get_image_bytes, get_raw_image_data
from src.utils.printer import pprint as print
from src.logger.logger import logger

2025-05-25 12:15:45,853 - INFO - Anonymized telemetry enabled. See https://docs.browser-use.com/development/telemetry for more information.


In [3]:
# --- file config.py
class Config:
    ENDPOINT: Path = __root__ /'SANDBOX' / 'davidka'
    SUPPLIERS_ENDPOINT: Path = __root__ / 'src' / 'suppliers' / 'suppliers_list'
    config:SimpleNamespace = j_loads_ns(ENDPOINT / 'davidka.json')
    GEMINI_API_KEY:str = gs.credentials.gemini.onela.api_key
    PRESTA_API_KEY:str = gs.credentials.prestashop.store_davidka_net.api_key
    PRESTA_DOMAIN:str = gs.credentials.prestashop.store_davidka_net.api_domain
    gemini_model_name:str = config.gemini_model_name
    system_instruction:str = ' ' # <- Это пробел!
    webdriver_window_mode:str = 'headless'
# --- end file config.pt

In [4]:
''' Здесь тестирую функции `ProductFields`



async def description_short(self, value:Optional[str] = '') -> bool:
    """Fetch and set short description.
    
    Args:
    value (atr): это значение можно передать в словаре kwargs через ключ {description_short = `value`} при определении класса.
    Если `value` было передано, его значение подставляется в поле `ProductFields.description_short`.
    """

    try:
        # Получаем значение через execute_locator
        value =  value or await self.driver.execute_locator(self.product_locator.description_short)
        if not value:
            ...
            return
        self.fields.description_short = normalize_string()
        return True

    except Exception as ex:
        logger.error(f"Ошибка получения значения в поле `description_short`", ex)
        ...
        return

    self.fields.description_short = value
    return True

In [ ]:
async def get_list_products_in_category (d: Driver, l: SimpleNamespace) -> list:    

    product_links: List[str] | str | None = await d.execute_locator(l.product_links)
    return product_links if isinstance(product_links, list) else [product_links] if product_links else None # здесь условие, что product_links строка и не только



In [5]:

class Scenario:
    """Dataclass for designing and promoting images through various platforms."""

    gemini: Optional[GoogleGenerativeAi] = None
    openai: Optional[OpenAIModel] = None
    product: PrestaProduct = None
    driver: Driver = None
    graber: Graber = None

    def __init__(self,
            presta_api_key:Optional[str] = '',
            presta_api_domain:Optional[str] = '',
            gemini_model_name:Optional[str] = '',
            openai_model_name:Optional[str] = '',
            gemini_api_key:Optional[str] = '',
            openai_api_key:Optional[str] = '',
            gemini: Optional[GoogleGenerativeAi] = None, 
            openai: Optional[OpenAIModel] = None,
            system_instruction:str = '',
            driver:Driver = None, 
            webdriver_window_mode:str = ''
            ):
        """
        Инициализация 
            Args:
                presta_api_key:Optional[str] = '',
                presta_api_domain:Optional[str] = '',
                gemini_model_name:Optional[str] = '',
                openai_model_name:Optional[str] = '',
                gemini_api_key:Optional[str] = '',
                openai_api_key:Optional[str] = '',
                gemini: Optional[GoogleGenerativeAi] = None, 
                openai: Optional[OpenAIModel] = None,
        """
        ...
        
        self.driver = driver or pydoll_driver

        if gemini:
            self.gemini = gemini
        else:
            gemini_api_key:str = gemini_api_key if gemini_api_key else Config.GEMINI_API_KEY
            gemini_model_name:str = gemini_model_name if gemini_model_name else Config.gemini_model_name
            system_instruction:str = system_instruction if system_instruction else Config.system_instruction
            if not self._init_gemini(gemini_api_key, gemini_model_name, system_instruction):
                logger.debug('Модель GEMINI не иницаилизирована')
                

        presta_api_key:str = presta_api_key if presta_api_key else Config.PRESTA_API_KEY
        presta_api_domain:str = presta_api_domain if presta_api_domain else Config.PRESTA_DOMAIN
        if not presta_api_key or not presta_api_domain:
            logger.critical(f'Проверь \nAPI {presta_api_key}\nDomain {presta_api_domain=}')
            return False

        self.product = PrestaProduct(presta_api_key, presta_api_domain )

    def _init_gemini(self, api_key: str, model_name: str, system_instruction: str) -> bool:
        """"""
        try:
            generation_config = dict({'response_mime_type':'application/json'})
            self.gemini = GoogleGenerativeAi(api_key, model_name, generation_config, system_instruction)
            return True
        except Exception as ex:
            logger.error(f'Ошибка иницализации модели!', ex, False)
            return False


    async def process_supplier(self, supplier_prefix:str) -> bool:
        """"""
        ...
        try:
            supplier_path:Path = Config.SUPPLIERS_ENDPOINT / supplier_prefix 
            self.graber = get_graber_by_supplier_prefix(self.driver, supplier_prefix)
            scenarios_list: list[dict] = j_loads(Config.SUPPLIERS_ENDPOINT / supplier_prefix / 'scenarios')
            locators_path:Path = supplier_path / 'locators' 
            locator_product:SimpleNamespace = j_loads_ns(locators_path / 'product.json')
            locator_category:SimpleNamespace = j_loads_ns(locators_path / 'category.json')
            categories_crawler:Any = None
            categories_crawler_module_path:str = f"src.suppliers.suppliers_list.{supplier_prefix}.categories_crawler"
        except Exception as ex:
            logger.error(f'Непредвиденная ошибка', ex)
            return False

        try:
            categories_crawler = importlib.import_module(categories_crawler_module_path)
        except Exception as ex:
            logger.error(f"Failed to import module `categories_crawler` '{supplier_prefix}'", ex)
            return False
        
        for scenario in scenarios_list:
            for _, item in scenario.items():
                self.driver.get_url(item['url'])
    
                products_urls_in_category:list = await get_list_products_in_category(self.driver, locator_category)
                if not products_urls_in_category:

                    continue # <- мб пустаая категория
                for product_url in products_urls_in_category:
                    self.driver.get_url(product_url)

                    # Не все поля товара надо заполнять. Вот кортеж необходимых полей:
                    actual_fields:tuple = ('id_manufacturer',
                                        'id_supplier',
                                        'name',                                                
                                        'description',
                                        'description_short',
                                        'default_image_url',
                                        )
                    self.graber.description_short = description_short
                    product_fields:ProductFields = await self.graber.grab_page_async(*actual_fields)
                    ipdb.set_trace()
                ...

            
    async def process_suppliers_list(self, suppliers_prefixes: str|list) -> bool:
        """
        Process suppliers based on the provided prefix.
        Args:
            suppliers_prefixes (Optional[str | List[str, str]], optional): Prefix for suppliers. Defaults to ''.
        Returns:
            bool: True if processing is successful, False otherwise.
        Raises:
            Exception: If any error occurs during supplier processing.
        """
        
        for supplier_prefix in suppliers_prefixes:
            try:
                await self.process_supplier(supplier_prefix)
            except Exception as ex:
                logger.error(f'Error while processing suppliers: {ex}')
                continue

In [ ]:

scenario: Scenario = Scenario(driver = driver)
suppliers_prefixes_list:list = ['ads_tec_iit_com',''] 
products_urls_in_category:list = [] 
product_fields:ProductFields = None

#await scenario.process_suppliers_list(suppliers_prefixes_list)
await scenario.process_supplier('ads_tec_iit_com')